In [388]:
from pydantic import BaseModel, Field, field_validator, model_validator, AliasChoices
from typing import Optional, Any
from datetime import datetime
import re

class Player(BaseModel):
    first_name: str = Field(validation_alias=AliasChoices("firstName", "first_name"))
    last_name: str = Field(validation_alias=AliasChoices("lastName", "last_name"))
    grad_class: int = Field(alias="class", validation_alias=AliasChoices("graduatingClass", "class","grad_class"))
    height: float = 0.0  # inches
    weight: float = 0.0
    scouting_report: Optional[str] = None
    maxpreps_career_id: Optional[str] = None
    id_247: Optional[str] = None
    base_player_id: Optional[str] = Field(None,validate_default=True)

    @field_validator("base_player_id", mode="after")
    @classmethod
    def gen_id(cls, v, info):
        if v: 
            return v
        
        # Pulling from validated data
        fn = info.data.get("first_name", "unknown")
        ln = info.data.get("last_name", "unknown")
        gc = info.data.get("grad_class", 0)
        mpid = info.data.get("maxpreps_career_id", "ABC")
        raw_id = f"{fn}{ln}{gc}{mpid}".lower().replace(" ", "")
        return re.sub(r'[^a-z0-9]', '', raw_id) 
        
    @field_validator("height", mode="before")
    @classmethod
    def parse_height(cls, v):
        if isinstance(v, str):
            # find all digits with optional . and optional numbers after that dot
            height_lst = [n for n in re.findall(r'\d+\.?\d*', v)]
            height_lst = [float(p) for p in height_lst if p] # filter out empty str
            if len(height_lst) <= 1:
                v = height_lst[0]
            else:
                v = (height_lst[0] * 12) + height_lst[1]

        return float(v) if v > 12 else float(v * 12)

    @field_validator("weight", mode="before")
    @classmethod
    def parse_weight(cls, w):
        if isinstance(w, str):
            weight_lst = [n for n in re.findall(r'\d+\.?\d*', w)]
            weight_lst = [float(p) for p in weight_lst if p]
            if re.search("kg", w, re.I):
                w = 2.2 * weight_lst[0]
            else:
                w = weight_lst[0]
        return w

class Team(BaseModel):
    sport: str
    mascot: Optional[str] = None
    school_name: Optional[str] = None
    city: Optional[str] = None
    state: Optional[str] = None
    maxpreps_url: Optional[str] = Field(None, validation_alias=AliasChoices("team_canonical_url", "teamCanonicalUrl","maxpreps_url"))
    maxpreps_team_id: Optional[str] = Field(None, validation_alias=AliasChoices("team_id", "teamId","maxpreps_team_id"))
    team_id: Optional[str] = Field(None,validate_default=True)
    @model_validator(mode='before')
    @classmethod
    def parse_from_url(cls, data: dict):
        url = data.get("maxpreps_url")
        if url and isinstance(url, str):
            # Format: "/state/city/school-mascot/sport/"
            parts = [p for p in url.split("/") if p]
            
            # Populate missing fields from URL parts
            if len(parts) >= 3:
                data["state"] = data.get("state") or parts[0].lower()
                data["city"] = data.get("city") or parts[1].replace("-", " ").title()
                name_parts = parts[2].split("-")
                
                if len(name_parts) > 1:
                    # [:-1] = "Rainier Beach", [-1] = "Vikings"
                    data["school_name"] = " ".join(name_parts[:-1]).title()
                    data["mascot"] = name_parts[-1].title()
                else:
                    # Fallback if there is no hyphen
                    data["school_name"] = name_parts[0].title()
        return data

    @field_validator("team_id", mode="after")
    @classmethod
    def gen_id(cls, v, info):
        if v: 
            return v
        
        # Pulling from validated data
        msct = info.data.get("mascot", "unknown")
        schl = info.data.get("school_name", "unknown")
        state = info.data.get("state", 0)
        mptid = info.data.get("maxpreps_team_id", "ABC")
        raw_id = f"{msct}{schl}{state}{mptid}".lower().replace(" ", "")
        return re.sub(r'[^a-z0-9]', '', raw_id) 
    

class Game(BaseModel):
    contest_id: str
    date: datetime
    opponent_name: str
    opponent_id: Optional[str] = None
    is_home: bool = True
    
    # Contest Result
    result: Optional[str] = None  # 'W' or 'L'
    team_score: int = 0
    opponent_score: int = 0
    
    # Player's Box Score for this specific game
    minutes_played: int = 0
    points: int = 0
    rebounds: int = 0
    assists: int = 0
    steals: int = 0
    blocks: int = 0
    turnovers: int = 0
    fouls: int = 0
    
    fg_made: int = 0
    fg_attempted: int = 0
    fg3_made: int = 0
    fg3_attempted: int = 0
    ft_made: int = 0
    ft_attempted: int = 0

class BasketballRecord(Player):
    # Context & Foreign Keys
    position: Optional[str] = None
    jersey: Optional[str] = None
    team_id: str 
    
    # --- PER GAME AVERAGES ---
    games_played: int = 0
    minutes_per_game: float = 0.0
    points_per_game: float = 0.0
    off_rebounds_per_game: float = 0.0
    def_rebounds_per_game: float = 0.0
    rebounds_per_game: float = 0.0
    assists_per_game: float = 0.0
    steals_per_game: float = 0.0
    blocks_per_game: float = 0.0
    turnovers_per_game: float = 0.0
    fouls_per_game: float = 0.0

    # --- SEASON TOTALS ---
    minutes_played: int = 0
    points: int = 0
    off_rebounds: int = 0
    def_rebounds: int = 0
    rebounds: int = 0
    assists: int = 0
    steals: int = 0
    blocks: int = 0
    turnovers: int = 0
    fouls: int = 0

    # --- SHOOTING ---
    fg_made: int = 0
    fg_attempted: int = 0
    fg_pct: float = 0.0
    
    fg2_made: int = 0
    fg2_attempted: int = 0
    fg2_pct: float = 0.0

    fg3_made: int = Field(0, alias="ThreePointsMade")
    fg3_attempted: int = Field(0, alias="ThreePointAttempts")
    fg3_pct: float = Field(0.0, alias="ThreePointPercentage")
    
    ft_made: int = 0
    ft_attempted: int = 0
    ft_pct: float = 0.0
    
    points_per_shot: float = 0.0
    efg_pct: float = 0.0

    # --- ADVANCED / MISC ---
    ast_to_ratio: float = 0.0
    stl_to_ratio: float = 0.0
    stl_pf_ratio: float = 0.0
    blk_pf_ratio: float = 0.0
    charges: int = 0
    deflections: int = 0
    tech_fouls: int = 0
    double_doubles: int = 0
    triple_doubles: int = 0
    
    @model_validator(mode='before')
    @classmethod
    def clean_empty_strings(cls, data: Any) -> Any:
        if isinstance(data, dict):
            # List of keys that are intended to be text/strings
            string_fields = {"position", "jersey", "school_name", "city", "state", "base_player_id"}
            
            for key, value in data.items():
                if value == "" or value is None:
                    if key in string_fields:
                        data[key] = None # Safe for Optional[str]
                    else:
                        data[key] = 0    # Safe for int/float
        return data

In [389]:
def traverse_paths(json_blob, value_map_json):
    new_dict = {}
    for key, path in value_map_json.items():
        current_node = json_blob
        try:
            for node in path:
                # If current_node is a list and path instruction is a dict, use it as a filter
                if isinstance(current_node, list) and isinstance(node, dict):
                    f_key, f_val = next(iter(node.items()))
                    # Find the first dictionary in the list that matches the key/value pair
                    current_node = next(item for item in current_node if item.get(f_key) == f_val)
                else:
                    # Standard key/index traversal
                    current_node = current_node[node]
            new_dict[key] = current_node
            
        except (KeyError, TypeError, IndexError, StopIteration):
            # Fails safely if the stat doesn't exist for a specific player
            new_dict[key] = None
            
    return new_dict

In [390]:
def extract_all_teams(json_blob, team_mapping):
    # 1. Get the path to the list of sports
    root_key = next(iter(team_mapping)) # "team_root"
    root_path = team_mapping[root_key]
    
    #returns all teams as a dict to root_key, confusing name 
    raw_sports_list = traverse_paths(json_blob, {root_key: root_path})[root_key] 
    
    if not raw_sports_list:
        return []
    return raw_sports_list

def parse_into_teams(teams_list,team_struct):
    all_teams = []

    for i in range(len(teams_list)):
        team_dict = traverse_paths(teams_list[i],team_struct)
        all_teams.append(Team(**team_dict))
    return all_teams

In [391]:
path = "PlayerStats/Stokes2.json"
value_maps = "player_val_mappings.json"
with open(path, "r") as f:
    # Use json.load() (NO 'S') to read directly from a file object
    data = json.load(f)
with open(value_maps, "r") as f:
    maps = json.load(f)
bio_maps = maps["maxpreps-stats"]
stat_maps =maps["maxpreps-basketball-stats"] #should have a key function
team_maps = maps["team_mapping"]

In [406]:
new_vals = traverse_paths(data,bio_maps)
vals = traverse_paths(data,stat_maps)
teams = extract_all_teams(data, team_maps)
teams_obj = parse_into_teams(teams,maps["team_structure"])
recent_team_val = teams[0]|teams_obj[0].model_dump()

In [407]:
recent_team_val

{'teamId': 'd1ded3d1-2ebc-4b4b-94f2-e397086569eb',
 'athleteId': 'd9bfc4ca-c8f6-4439-a22a-63479ff81698',
 'classYear': 'Senior',
 'gender': 'Boys',
 'jersey': '4',
 'positions': ['G'],
 'sport': 'Basketball',
 'sportSeasonId': 'ee7cf537-7394-4430-88ca-becaabda286c',
 'teamCanonicalUrl': '/wa/seattle/rainier-beach-vikings/basketball/',
 'teamHomeCanonicalUrl': '/wa/seattle/rainier-beach-vikings/basketball/',
 'teamLevel': 'Varsity',
 'mascot': 'Vikings',
 'school_name': 'Rainier Beach',
 'city': 'Seattle',
 'state': 'wa',
 'maxpreps_url': '/wa/seattle/rainier-beach-vikings/basketball/',
 'maxpreps_team_id': 'd1ded3d1-2ebc-4b4b-94f2-e397086569eb',
 'team_id': 'vikingsrainierbeachwad1ded3d12ebc4b4b94f2e397086569eb'}

In [408]:
print(new_vals)

{'first_name': 'Tyran', 'last_name': 'Stokes', 'grad_class': 2026, 'height_ft': 6, 'height_in': 8, 'weight': 230, 'maxpreps_career_id': 'd32ac4c6-f83b-48a4-88d4-1eaae652525c'}


In [409]:
print(vals)

{'games_played': '29', 'minutes_per_game': '', 'points_per_game': '21.0', 'def_rebounds_per_game': '7.4', 'off_rebounds_per_game': '1.9', 'rebounds_per_game': '9.3', 'assists_per_game': '3.9', 'steals_per_game': '1.5', 'blocks_per_game': '0.8', 'turnovers_per_game': '3.8', 'fouls_per_game': '2.3', 'minutes_played': '', 'points': '610', 'fg_made': '203', 'fg_attempted': '378', 'fg_pct': '54', 'points_per_shot': '1.6', 'efg_pct': '57', 'fg3_made': '23', 'fg3_attempted': '76', 'fg3_pct': '30', 'ft_made': '181', 'ft_attempted': '277', 'ft_pct': '65', 'fg2_made': '180', 'fg2_attempted': '302', 'fg2_pct': '60', 'off_rebounds': '55', 'def_rebounds': '216', 'rebounds': '271', 'assists': '112', 'steals': '44', 'blocks': '23', 'turnovers': '111', 'fouls': '68', 'ast_to_ratio': '1.01', 'stl_to_ratio': '0.40', 'stl_pf_ratio': '0.65', 'blk_pf_ratio': '0.34', 'charges': '1', 'deflections': '48', 'tech_fouls': '', 'double_doubles': '17', 'triple_doubles': ''}


In [410]:
team_val

[{'teamId': 'd1ded3d1-2ebc-4b4b-94f2-e397086569eb',
  'athleteId': 'd9bfc4ca-c8f6-4439-a22a-63479ff81698',
  'classYear': 'Senior',
  'gender': 'Boys',
  'jersey': '4',
  'positions': ['G'],
  'sport': 'Basketball',
  'sportSeasonId': 'ee7cf537-7394-4430-88ca-becaabda286c',
  'teamCanonicalUrl': '/wa/seattle/rainier-beach-vikings/basketball/',
  'teamHomeCanonicalUrl': '/wa/seattle/rainier-beach-vikings/basketball/',
  'teamLevel': 'Varsity'}]

In [412]:
combined_vals = new_vals|vals|recent_team_val
combined_vals["height"]= 12* combined_vals["height_ft"] + combined_vals["height_in"]

In [413]:
p=BasketballRecord(**(combined_vals))

In [415]:
p.model_dump()

{'first_name': 'Tyran',
 'last_name': 'Stokes',
 'grad_class': 2026,
 'height': 80.0,
 'weight': 230.0,
 'scouting_report': None,
 'maxpreps_career_id': 'd32ac4c6-f83b-48a4-88d4-1eaae652525c',
 'id_247': None,
 'base_player_id': 'tyranstokes2026d32ac4c6f83b48a488d41eaae652525c',
 'position': None,
 'jersey': '4',
 'team_id': 'vikingsrainierbeachwad1ded3d12ebc4b4b94f2e397086569eb',
 'games_played': 29,
 'minutes_per_game': 0.0,
 'points_per_game': 21.0,
 'off_rebounds_per_game': 1.9,
 'def_rebounds_per_game': 7.4,
 'rebounds_per_game': 9.3,
 'assists_per_game': 3.9,
 'steals_per_game': 1.5,
 'blocks_per_game': 0.8,
 'turnovers_per_game': 3.8,
 'fouls_per_game': 2.3,
 'minutes_played': 0,
 'points': 610,
 'off_rebounds': 55,
 'def_rebounds': 216,
 'rebounds': 271,
 'assists': 112,
 'steals': 44,
 'blocks': 23,
 'turnovers': 111,
 'fouls': 68,
 'fg_made': 203,
 'fg_attempted': 378,
 'fg_pct': 54.0,
 'fg2_made': 180,
 'fg2_attempted': 302,
 'fg2_pct': 60.0,
 'fg3_made': 0,
 'fg3_attempted'

In [ ]:
##MAXPREPS PLAYER OBJ RETRIEVAL FOR 

In [ ]:
##247 PARSING AND STORING INTO OBJ

In [105]:
##WE NOW HAVE A WAY TO GRAB FROM 247 HEADERS. we can use this to grab stats blobs.

pathOfInterest = 'PlayerIndices/2026PlayerIndexHTML.txt'
with open(pathOfInterest, 'r') as f:
    player_list = f.read().split("\n")
soup = BeautifulSoup(player_list[0], 'html.parser')
target_elements = soup.find_all('div', class_=['recruit', 'metrics', 'position'])
target_elements
pieces = [s for elem in target_elements for s in list(elem.stripped_strings)]##OUTER LOOP MUST COME FIRST
#s for (outer loop) for s in list(xxx)... s(first one) is main character.
print(pieces)

['Tyran Stokes', 'Rainier Beach (Seattle, WA)', 'SF', '6-7 / 230']
